In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from Bio.Seq import Seq
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()

from peint.data.datamodule import PLMRDataModule
from peint.models.frameworks.peint import load_from_new_checkpoint

from evo.sequence import get_mutant
from evo.dms import get_site_by_site_consensus
from evo.dataset import CherriesDataset, EncodedPEINTDataset

/accounts/projects/yss/stephen.lu/peint/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# Load trained models from checkpoints
ckpt_dir = Path("/accounts/projects/yss/stephen.lu/protevo/plmr/logs/train/runs")
ckpt_paths = {
    "heavy": ckpt_dir / "2025-08-08_15-36-04/checkpoints/epoch_089.ckpt",
    "light": ckpt_dir / "2025-08-10_11-57-18/checkpoints/epoch_042.ckpt",
    "joint": ckpt_dir / "2025-08-10_11-58-46/checkpoints/epoch_044.ckpt",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

modules = {k: load_from_new_checkpoint(v, device) for k, v in ckpt_paths.items()}
vocab = next(iter(modules.values())).net.vocab

Using device: cuda


/accounts/projects/yss/stephen.lu/peint/peint/models/frameworks/peint.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locat

In [11]:
# Calculate likelihood of leaves conditioned on the root and distance from root
import tempfile

def dataloader_from_transitions(transitions, batch_size=32, mask_prob=0.0):
    datafile = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(datafile.name, "w") as f:
        f.write("{0} transitions\n".format(len(transitions)))
        f.write("\n".join(transitions))

    dataset = EncodedPEINTDataset(
        dataset=CherriesDataset(data_file=datafile.name),
        vocab=vocab,
        mask_prob=mask_prob,
        random_token_prob=0.0,
        leave_unmasked_prob=0.0,
    )
    generator = iter(
        PLMRDataModule(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=False,
        )._dataloader_template(dataset=dataset, training=False)
    )
    return generator

def infer_log_likelihoods(dataloader, module):
    # run inference on the dataloader
    lls = []
    for batch in tqdm(dataloader, desc="Inference"):
        batch = [b.to(device) for b in batch]
        [x, x_targets, y, y_targets, t, x_pad_mask, y_pad_mask] = batch
        yt_mask = y_targets != module.net.vocab.pad_idx  # actual values

        with torch.no_grad() and torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_logits, y_logits = module(x, y, t, x_pad_mask, y_pad_mask)

        y_logits = y_logits - torch.logsumexp(y_logits, dim=-1, keepdim=True)
        y_logits = y_logits.transpose(-1, -2)
        nll = F.cross_entropy(y_logits, y_targets, ignore_index=vocab.pad_idx, reduction="none")

        ll = -nll * yt_mask.float()
        ll = ll.sum(dim=-1)
        lls.append(ll.detach().cpu().numpy())

    lls = np.concatenate(lls)
    return lls

In [9]:
df = pd.read_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_er.csv")
# df = pd.read_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_Kd.csv")

heavy_wt = get_site_by_site_consensus(df, "heavy")
light_wt = get_site_by_site_consensus(df, "light")
print(len(heavy_wt), len(light_wt), len(heavy_wt) + len(light_wt))

df['heavy_mut'] = df['heavy'].apply(lambda x: get_mutant(x, heavy_wt))
df['light_mut'] = df['light'].apply(lambda x: get_mutant(x, light_wt))
df['mut'] = df['heavy_mut'] + df['light_mut']

# df = df[['heavy', 'light', 'fitness', 'mut']]
# df.to_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_er.csv", index=False)

120 108 228


In [48]:
df_heavy = df[df["heavy_mut"] != ""]
df_heavy[['fitness', 'mut']].to_csv("/accounts/projects/yss/stephen.lu/peint/data/wyatt/dms/koenig_kd_heavy.csv", index=False)
df_light = df[df["light_mut"] != ""]
df_light[['fitness', 'mut']].to_csv("/accounts/projects/yss/stephen.lu/peint/data/wyatt/dms/koenig_kd_light.csv", index=False)

In [5]:
def join_chains(heavy: str, light: str) -> str:
    return heavy + "G" * 10 + light

# def join_chains(heavy: str, light: str) -> str:
    # return heavy + "." + light

In [6]:
joint_heavy_transitions = [
    f"{join_chains(heavy_wt, light_wt)} {join_chains(row['heavy'], light_wt)} {row['mut']}" for i, row in df.iterrows() if row['heavy'] != heavy_wt
]
joint_light_transitions = [
    f"{join_chains(heavy_wt, light_wt)} {join_chains(heavy_wt, row['light'])} {row['mut']}" for i, row in df.iterrows() if row['light'] != light_wt
]
joint_transitions = joint_heavy_transitions + joint_light_transitions
heavy_transitions = [
    f"{heavy_wt} {row['heavy']} {row['mut']}" for i, row in df.iterrows() if row['heavy'] != heavy_wt
]
light_transitions = [
    f"{light_wt} {row['light']} {row['mut']}" for i, row in df.iterrows() if row['light'] != light_wt
]
print(len(joint_transitions), len(joint_heavy_transitions), len(joint_light_transitions), len(heavy_transitions), len(light_transitions))

4275 2261 2014 2261 2014


In [ ]:
data_dir = Path("/accounts/projects/yss/stephen.lu/peint/data/wyatt/dms")

with open(data_dir / "mut_joint_pipet.txt", "w") as f:
    f.write("{0} transitions\n".format(len(joint_transitions)))
    f.write("\n".join(joint_transitions))

with open(data_dir / "mut_heavy.txt", "w") as f:
    f.write("{0} transitions\n".format(len(heavy_transitions)))
    f.write("\n".join(heavy_transitions))

with open(data_dir / "mut_light.txt", "w") as f:
    f.write("{0} transitions\n".format(len(light_transitions)))
    f.write("\n".join(light_transitions))

In [ ]:
heavy_transitions = [
    f"{heavy_wt.strip()} {row.heavy.strip()} 7.0" for _, row in df.iterrows()
]
dataloader = dataloader_from_transitions(heavy_transitions, batch_size=32)
heavy_lls = infer_log_likelihoods(dataloader, modules["heavy"])

Inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 134/134 [00:21<00:00,  6.34it/s]


In [12]:
light_transitions = [
    f"{light_wt.strip()} {row.light.strip()} 7.0" for _, row in df.iterrows()
]
dataloader = dataloader_from_transitions(light_transitions, batch_size=32)
light_lls = infer_log_likelihoods(dataloader, modules["light"])

Inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 134/134 [00:10<00:00, 12.49it/s]


In [14]:
muts = []
joint_transitions = []

for i, row in df.iterrows():
    if row['heavy'] != heavy_wt:
        muts.append(row['heavy_mut'])
        joint_transitions.append(f"{join_chains(heavy_wt, light_wt)} {join_chains(row['heavy'], light_wt)} 7.0")

    if row['light'] != light_wt:
        muts.append(row['light_mut'])
        joint_transitions.append(f"{join_chains(heavy_wt, light_wt)} {join_chains(heavy_wt, row['light'])} 7.0")

dataloader = dataloader_from_transitions(joint_transitions, batch_size=32)
joint_lls = infer_log_likelihoods(dataloader, modules["joint"])

print(len(muts), len(joint_lls))

Inference: 100%|██████████████████████████████████████████████████████████████| 134/134 [00:12<00:00, 11.04it/s]

4275 4275


In [15]:
df['heavy_ll'] = heavy_lls
df['light_ll'] = light_lls
df['joint_ll'] = df['heavy_ll'] + df['light_ll']
df['ppl'] = np.exp(-df['joint_ll'] / (len(heavy_wt) + len(light_wt)))

In [15]:
df.set_index('mut', inplace=True)
for mut, joint_ll in zip(muts, joint_lls):
    df.at[mut, 'joint_ll'] = joint_ll
    df.at[mut, 'ppl'] = np.exp(-joint_ll / (len(heavy_wt) + len(light_wt)))

In [18]:
df.to_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_peint_joint_t=7.csv", index=False)